In [1]:
pip install pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd

df = pd.read_csv(
    "training_set_rel3.tsv",
    sep="\t",
    encoding="latin1"
)

print(df.shape)
print(df.columns)

(12976, 28)
Index(['essay_id', 'essay_set', 'essay', 'rater1_domain1', 'rater2_domain1',
       'rater3_domain1', 'domain1_score', 'rater1_domain2', 'rater2_domain2',
       'domain2_score', 'rater1_trait1', 'rater1_trait2', 'rater1_trait3',
       'rater1_trait4', 'rater1_trait5', 'rater1_trait6', 'rater2_trait1',
       'rater2_trait2', 'rater2_trait3', 'rater2_trait4', 'rater2_trait5',
       'rater2_trait6', 'rater3_trait1', 'rater3_trait2', 'rater3_trait3',
       'rater3_trait4', 'rater3_trait5', 'rater3_trait6'],
      dtype='str')


In [3]:
print(df[['essay_set', 'essay', 'domain1_score']].head(3).to_string())

   essay_set                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            

In [4]:
print(df['essay_set'].value_counts().sort_index())

essay_set
1    1783
2    1800
3    1726
4    1770
5    1805
6    1800
7    1569
8     723
Name: count, dtype: int64


In [5]:
print(df[['essay_set', 'domain1_score']].groupby('essay_set').agg(['min', 'max', 'mean']))

          domain1_score               
                    min max       mean
essay_set                             
1                     2  12   8.528323
2                     1   6   3.415556
3                     0   3   1.848204
4                     0   3   1.432203
5                     0   4   2.408864
6                     0   4   2.720000
7                     2  24  16.062460
8                    10  60  36.950207


In [6]:
for essay_set in sorted(df['essay_set'].unique()):
    print("Essay Set:", essay_set)
    print(df[df['essay_set'] == essay_set]['essay'].iloc[0][:500])
    print("-" * 80)

Essay Set: 1
Dear local newspaper, I think effects computers have on people are great learning skills/affects because they give us time to chat with friends/new people, helps us learn about the globe(astronomy) and keeps us out of troble! Thing about! Dont you think so? How would you feel if your teenager is always on the phone with friends! Do you ever time to chat with your friends or buisness partner about things. Well now - there's a new way to chat the computer, theirs plenty of sites on the internet to
--------------------------------------------------------------------------------
Essay Set: 2
Certain materials being removed from libraries such as books, music and magazines, shouldn't be removed from the libraries. It gives people a chance to understand how the real world @CAPS2.     Having certain materials such as books and music definitly should not be removed, because most books and music can show most people how bad the statement in the book @CAPS2 or how bad the lyrics are

In [7]:
print(df.isnull().sum())

essay_id              0
essay_set             0
essay                 0
rater1_domain1        0
rater2_domain1        0
rater3_domain1    12848
domain1_score         0
rater1_domain2    11176
rater2_domain2    11176
domain2_score     11176
rater1_trait1     10684
rater1_trait2     10684
rater1_trait3     10684
rater1_trait4     10684
rater1_trait5     12253
rater1_trait6     12253
rater2_trait1     10684
rater2_trait2     10684
rater2_trait3     10684
rater2_trait4     10684
rater2_trait5     12253
rater2_trait6     12253
rater3_trait1     12848
rater3_trait2     12848
rater3_trait3     12848
rater3_trait4     12848
rater3_trait5     12848
rater3_trait6     12848
dtype: int64


In [8]:
print("Missing essays:", df['essay'].isnull().sum())

Missing essays: 0


In [9]:
print("Duplicate essays:", df['essay'].duplicated().sum())

Duplicate essays: 4


In [10]:
import re

def clean_essay(text):
    # Remove anonymization tags such as @CAPS1, @PERSON2, @DATE1
    text = re.sub(r'@\w+\d*', '', text)
    
    # Replace multiple spaces/newlines with a single space
    text = re.sub(r'\s+', ' ', text)
    
    # Remove leading/trailing spaces
    text = text.strip()
    
    return text

df['clean_essay'] = df['essay'].apply(clean_essay)

print(df[['essay', 'clean_essay']].head(3).to_string())

In [11]:
print(df[['essay', 'clean_essay']].iloc[0].to_string())

essay          Dear local newspaper, I think effects computer...
clean_essay    Dear local newspaper, I think effects computer...


In [12]:
import re

def extract_features(text):
    sentences = re.split(r'[.!?]+', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    
    words = text.split()
    
    return {
        'word_count': len(words),
        'sentence_count': len(sentences),
        'avg_sentence_length': len(words) / len(sentences) if sentences else 0,
        'unique_word_count': len(set(word.lower() for word in words)),
        'vocabulary_richness': len(set(word.lower() for word in words)) / len(words) if words else 0
    }

features = df['clean_essay'].apply(extract_features).apply(pd.Series)

df = pd.concat([df, features], axis=1)

print(df[['essay_set', 'word_count', 'sentence_count',
          'avg_sentence_length', 'unique_word_count',
          'vocabulary_richness']].head())

   essay_set  word_count  sentence_count  avg_sentence_length  \
0          1       337.0            16.0            21.062500   
1          1       413.0            20.0            20.650000   
2          1       273.0            14.0            19.500000   
3          1       488.0            27.0            18.074074   
4          1       462.0            30.0            15.400000   

   unique_word_count  vocabulary_richness  
0              174.0             0.516320  
1              201.0             0.486683  
2              152.0             0.556777  
3              245.0             0.502049  
4              214.0             0.463203  


In [13]:
print(
    df[['word_count',
        'sentence_count',
        'avg_sentence_length',
        'unique_word_count',
        'vocabulary_richness',
        'domain1_score']].corr()['domain1_score'].sort_values(ascending=False)
)

domain1_score          1.000000
sentence_count         0.610159
unique_word_count      0.583974
word_count             0.576738
avg_sentence_length   -0.101790
vocabulary_richness   -0.396016
Name: domain1_score, dtype: float64


In [14]:
df['normalized_score'] = df.groupby('essay_set')['domain1_score'].transform(
    lambda x: (x - x.min()) / (x.max() - x.min())
)

print(df[['essay_set', 'domain1_score', 'normalized_score']].head(10))

   essay_set  domain1_score  normalized_score
0          1              8               0.6
1          1              9               0.7
2          1              7               0.5
3          1             10               0.8
4          1              8               0.6
5          1              8               0.6
6          1             10               0.8
7          1             10               0.8
8          1              9               0.7
9          1              9               0.7


In [15]:
print(
    df.groupby('essay_set')['normalized_score']
      .agg(['min', 'max', 'mean'])
)

           min  max      mean
essay_set                    
1          0.0  1.0  0.652832
2          0.0  1.0  0.483111
3          0.0  1.0  0.616068
4          0.0  1.0  0.477401
5          0.0  1.0  0.602216
6          0.0  1.0  0.680000
7          0.0  1.0  0.639203
8          0.0  1.0  0.539004


In [16]:
pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [17]:
from sklearn.model_selection import train_test_split

X = df['clean_essay']
y = df['normalized_score']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training essays:", len(X_train))
print("Testing essays:", len(X_test))

Training essays: 10380
Testing essays: 2596


In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=10000,
    stop_words='english',
    ngram_range=(1, 2)
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Training shape:", X_train_tfidf.shape)
print("Testing shape:", X_test_tfidf.shape)

Training shape: (10380, 10000)
Testing shape: (2596, 10000)


In [19]:
from sklearn.linear_model import Ridge

model = Ridge(alpha=1.0)

model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)

print("Model trained successfully!")
print("First 10 predictions:")
print(y_pred[:10])

Model trained successfully!
First 10 predictions:
[0.35465463 0.57114992 0.65441808 0.38063864 0.41119117 0.1606044
 0.42462884 0.86359344 0.79579993 0.6648511 ]


In [20]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
correlation = np.corrcoef(y_test, y_pred)[0, 1]

print("MAE:", mae)
print("RMSE:", rmse)
print("Correlation:", correlation)

MAE: 0.12590777225637576
RMSE: 0.16233453469236564
Correlation: 0.7397564009586368


In [21]:
!pip install textstat

Defaulting to user installation because normal site-packages is not writeable


In [22]:
import textstat

textstat.set_lang("en")

df['readability_score'] = df['clean_essay'].apply(
    lambda x: textstat.flesch_reading_ease(x)
)

print(df[['essay_set', 'word_count', 'readability_score', 'domain1_score']].head(10))

   essay_set  word_count  readability_score  domain1_score
0          1       337.0          72.264162              8
1          1       413.0          65.900744              9
2          1       273.0          65.191471              7
3          1       488.0          60.866218             10
4          1       462.0          67.916358              8
5          1       239.0          77.572464              8
6          1       491.0          67.199615             10
7          1       476.0          70.397176             10
8          1       439.0          74.521725              9
9          1       495.0          75.223689              9


In [23]:
correlation = df['readability_score'].corr(df['domain1_score'])

print("Readability vs Essay Score Correlation:", correlation)

Readability vs Essay Score Correlation: 0.30623084906262826


In [24]:
!pip install language-tool-python

Defaulting to user installation because normal site-packages is not writeable


In [25]:
import re

def grammar_features(text):
    sentences = re.split(r'[.!?]+', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    
    words = text.split()
    
    # Basic punctuation counts
    comma_count = text.count(',')
    question_count = text.count('?')
    exclamation_count = text.count('!')
    
    # Very long sentences
    long_sentences = sum(
        len(sentence.split()) > 30
        for sentence in sentences
    )
    
    # Repeated words
    word_list = [word.lower().strip(".,!?;:\"'()") for word in words]
    
    repeated_words = sum(
        word_list[i] == word_list[i-1]
        for i in range(1, len(word_list))
        if word_list[i]
    )
    
    return {
        'comma_count': comma_count,
        'question_count': question_count,
        'exclamation_count': exclamation_count,
        'long_sentence_count': long_sentences,
        'repeated_word_count': repeated_words
    }

grammar_df = df['clean_essay'].apply(grammar_features).apply(pd.Series)

df = pd.concat([df, grammar_df], axis=1)

print(df[
    ['essay_set', 'comma_count', 'question_count',
     'exclamation_count', 'long_sentence_count',
     'repeated_word_count', 'domain1_score']
].head(10))

   essay_set  comma_count  question_count  exclamation_count  \
0          1           18               2                  4   
1          1           12               1                  1   
2          1            9               0                  0   
3          1           13               1                  2   
4          1           13               0                  0   
5          1            3               2                  1   
6          1           35               4                  0   
7          1           11               3                  1   
8          1           16               6                  1   
9          1           14               2                  0   

   long_sentence_count  repeated_word_count  domain1_score  
0                    5                    0              8  
1                    3                    0              9  
2                    3                    1              7  
3                    3                    0        

In [26]:
feature_columns = [
    'word_count',
    'sentence_count',
    'avg_sentence_length',
    'unique_word_count',
    'vocabulary_richness',
    'readability_score',
    'comma_count',
    'question_count',
    'exclamation_count',
    'long_sentence_count',
    'repeated_word_count'
]

correlations = df[feature_columns + ['domain1_score']].corr()['domain1_score']
correlations = correlations.sort_values(ascending=False)

print(correlations)

domain1_score          1.000000
sentence_count         0.610159
unique_word_count      0.583974
word_count             0.576738
comma_count            0.412042
exclamation_count      0.320621
readability_score      0.306231
repeated_word_count    0.258212
long_sentence_count    0.242687
question_count         0.175218
avg_sentence_length   -0.101790
vocabulary_richness   -0.396016
Name: domain1_score, dtype: float64


In [27]:
from scipy.sparse import hstack
from sklearn.preprocessing import StandardScaler

numeric_features = [
    'word_count',
    'sentence_count',
    'avg_sentence_length',
    'unique_word_count',
    'vocabulary_richness',
    'readability_score',
    'comma_count',
    'question_count',
    'exclamation_count',
    'long_sentence_count',
    'repeated_word_count'
]

scaler = StandardScaler()

X_train_numeric = scaler.fit_transform(
    df.loc[X_train.index, numeric_features]
)

X_test_numeric = scaler.transform(
    df.loc[X_test.index, numeric_features]
)

X_train_combined = hstack([
    X_train_tfidf,
    X_train_numeric
])

X_test_combined = hstack([
    X_test_tfidf,
    X_test_numeric
])

print("Combined training shape:", X_train_combined.shape)
print("Combined testing shape:", X_test_combined.shape)

Combined training shape: (10380, 10011)
Combined testing shape: (2596, 10011)


In [28]:
from sklearn.linear_model import Ridge

combined_model = Ridge(alpha=1.0)

combined_model.fit(X_train_combined, y_train)

y_pred_combined = combined_model.predict(X_test_combined)

print("Combined model trained successfully!")
print("First 10 predictions:")
print(y_pred_combined[:10])

Combined model trained successfully!
First 10 predictions:
[0.37376534 0.58518617 0.61496387 0.36158843 0.43559805 0.15781872
 0.43947817 0.90996335 0.838716   0.71692438]


In [29]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae_combined = mean_absolute_error(y_test, y_pred_combined)
rmse_combined = np.sqrt(mean_squared_error(y_test, y_pred_combined))
correlation_combined = np.corrcoef(y_test, y_pred_combined)[0, 1]

print("Combined Model Results")
print("MAE:", mae_combined)
print("RMSE:", rmse_combined)
print("Correlation:", correlation_combined)

Combined Model Results
MAE: 0.11961206434404936
RMSE: 0.15473375152863575
Correlation: 0.7674771446688486


In [30]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

rf_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_numeric, y_train)

y_pred_rf = rf_model.predict(X_test_numeric)

mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
correlation_rf = np.corrcoef(y_test, y_pred_rf)[0, 1]

print("Random Forest Results")
print("MAE:", mae_rf)
print("RMSE:", rmse_rf)
print("Correlation:", correlation_rf)

Random Forest Results
MAE: 0.13833099723850878
RMSE: 0.1795815643685443
Correlation: 0.6680059639675296


In [31]:
def analyze_essay(essay):
    features = extract_features(essay)
    grammar = grammar_features(essay)

    return {
        **features,
        **grammar
    }

sample_essay = df['clean_essay'].iloc[0]

analysis = analyze_essay(sample_essay)

print(analysis)

{'word_count': 337, 'sentence_count': 16, 'avg_sentence_length': 21.0625, 'unique_word_count': 174, 'vocabulary_richness': 0.516320474777448, 'comma_count': 18, 'question_count': 2, 'exclamation_count': 4, 'long_sentence_count': 5, 'repeated_word_count': 0}


In [32]:
def generate_rule_feedback(features):
    feedback = []

    # Sentence structure
    if features['long_sentence_count'] > 3:
        feedback.append(
            "Some sentences are quite long. Consider splitting them "
            "into shorter sentences for better clarity."
        )

    # Vocabulary
    if features['vocabulary_richness'] < 0.35:
        feedback.append(
            "Your vocabulary is somewhat repetitive. Try using a wider "
            "range of words where appropriate."
        )
    else:
        feedback.append(
            "Your essay shows a good variety of vocabulary."
        )

    # Essay length
    if features['word_count'] < 200:
        feedback.append(
            "The essay is relatively short. Consider developing your "
            "ideas with more explanation or examples."
        )
    elif features['word_count'] > 700:
        feedback.append(
            "The essay is quite long. Consider removing unnecessary "
            "repetition and keeping the argument focused."
        )
    else:
        feedback.append(
            "The essay has a reasonable length for developing its ideas."
        )

    # Repeated words
    if features['repeated_word_count'] > 2:
        feedback.append(
            "Some words are repeated consecutively. Review these "
            "sentences for smoother writing."
        )

    # Questions
    if features['question_count'] > 3:
        feedback.append(
            "There are several questions in the essay. Make sure each "
            "one contributes to your main argument."
        )

    # Exclamation marks
    if features['exclamation_count'] > 3:
        feedback.append(
            "There are several exclamation marks. In formal essays, "
            "use them sparingly."
        )

    return feedback


feedback = generate_rule_feedback(analysis)

for i, item in enumerate(feedback, 1):
    print(f"{i}. {item}")

1. Some sentences are quite long. Consider splitting them into shorter sentences for better clarity.
2. Your essay shows a good variety of vocabulary.
3. The essay has a reasonable length for developing its ideas.
4. There are several exclamation marks. In formal essays, use them sparingly.


In [33]:
essay = df['clean_essay'].iloc[0]

prompt = f"""
You are an expert English writing evaluator.

Analyze the following student essay.

Evaluate it on these categories:
1. Grammar
2. Sentence Structure
3. Coherence and Flow
4. Argument Strength
5. Vocabulary
6. Overall Writing Quality

For each category:
- Give a score from 1 to 10
- Give short constructive feedback

Then provide:
- Overall score
- 3 specific improvement suggestions

Student Essay:
{essay}
"""

print(prompt)


You are an expert English writing evaluator.

Analyze the following student essay.

Evaluate it on these categories:
1. Grammar
2. Sentence Structure
3. Coherence and Flow
4. Argument Strength
5. Vocabulary
6. Overall Writing Quality

For each category:
- Give a score from 1 to 10
- Give short constructive feedback

Then provide:
- Overall score
- 3 specific improvement suggestions

Student Essay:
Dear local newspaper, I think effects computers have on people are great learning skills/affects because they give us time to chat with friends/new people, helps us learn about the globe(astronomy) and keeps us out of troble! Thing about! Dont you think so? How would you feel if your teenager is always on the phone with friends! Do you ever time to chat with your friends or buisness partner about things. Well now - there's a new way to chat the computer, theirs plenty of sites on the internet to do so: , , , facebook, myspace ect. Just think now while your setting up meeting with your boss o

In [34]:
import sys

print(sys.version)

3.14.7 (tags/v3.14.7:823f032, Aug  5 2026, 10:51:32) [MSC v.1944 64 bit (AMD64)]


In [35]:
%pip install openai

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [36]:
import openai

print("OpenAI package installed successfully!")

OpenAI package installed successfully!


In [37]:
import os

api_key = os.getenv("OPENAI_API_KEY")

print("API key loaded:", api_key is not None)

API key loaded: True


In [38]:
from openai import OpenAI

client = OpenAI()

print("OpenAI client connected successfully!")

OpenAI client connected successfully!


In [39]:
%pip install ollama

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [40]:
import ollama

print("Ollama Python package installed!")

Ollama Python package installed!


In [41]:
import ollama

response = ollama.chat(
    model='llama3.2:3b',
    messages=[
        {
            'role': 'user',
            'content': f"""
You are an expert English writing evaluator.

Analyze the following student essay.

Evaluate it on:
1. Grammar
2. Sentence Structure
3. Coherence and Flow
4. Argument Strength
5. Vocabulary
6. Overall Writing Quality

For each category:
- Give a score from 1 to 10
- Give short, constructive feedback

Then provide:
- Overall score
- 3 specific improvement suggestions

Student Essay:
{essay}
"""
        }
    ]
)

print(response['message']['content'])

**Grammar (Score: 4/10)**

The essay contains numerous grammatical errors, including:

* Missing or incorrect verb tenses (e.g., "I think effects" should be "I think effects")
* Incorrect subject-verb agreement (e.g., " Thing about! Dont you think so?" should be "Don't you think that?")
* Missing articles and prepositions (e.g., "how would you feel if your teenager is always on the phone with friends!" should be "how would you feel if your teenager were always on the phone with friends!")
* Run-on sentences and lack of punctuation

Constructive feedback: Review grammar rules and practice using correct verb tenses, subject-verb agreement, and punctuation.

**Sentence Structure (Score: 5/10)**

The essay features a mix of short and long sentences, but often lacks variety and transitions between ideas. Some sentences are sentence fragments or lack a clear subject or verb.

Constructive feedback: Vary sentence structure by using a range of sentence lengths, and incorporate transition words

In [42]:
import ollama

response = ollama.chat(
    model='llama3.2:3b',
    messages=[
        {
            'role': 'user',
            'content': f"""
You are an expert English writing evaluator.

Analyze this student essay and return ONLY valid JSON.

Use exactly this structure:

{{
  "grammar": {{
    "score": 0,
    "feedback": ""
  }},
  "sentence_structure": {{
    "score": 0,
    "feedback": ""
  }},
  "coherence": {{
    "score": 0,
    "feedback": ""
  }},
  "argument_strength": {{
    "score": 0,
    "feedback": ""
  }},
  "vocabulary": {{
    "score": 0,
    "feedback": ""
  }},
  "overall": {{
    "score": 0,
    "feedback": ""
  }},
  "suggestions": [
    "",
    "",
    ""
  ]
}}

Rules:
- Scores must be between 1 and 10.
- Feedback must be concise and constructive.
- Suggestions must be specific and actionable.
- Do not include Markdown.
- Do not include anything outside the JSON.

Student Essay:
{essay}
"""
        }
    ]
)

print(response['message']['content'])

{
  "grammar": {
    "score": 6,
    "feedback": "The essay contains several grammatical errors, including subject-verb agreement, tense consistency, and punctuation. For example, 'effects computers have on people are great learning skills' should be 'computers have great learning skills'."
  },
  "sentence_structure": {
    "score": 4,
    "feedback": "The essay has a lot of simple sentences and lacks varied sentence structures. Consider using compound or complex sentences to improve the flow of the essay."
  },
  "coherence": {
    "score": 7,
    "feedback": "The essay has a clear main idea, but the transitions between ideas could be smoother. Consider using transitional phrases to connect your ideas."
  },
  "argument_strength": {
    "score": 5,
    "feedback": "The essay presents a clear argument, but it could be stronger with more evidence and analysis. Consider incorporating more examples and counterarguments to build a more persuasive argument."
  },
  "vocabulary": {
    "sco

In [43]:
import json

feedback_text = response['message']['content']

feedback = json.loads(feedback_text)

print("JSON parsed successfully!")
print(feedback)

JSON parsed successfully!
{'grammar': {'score': 6, 'feedback': "The essay contains several grammatical errors, including subject-verb agreement, tense consistency, and punctuation. For example, 'effects computers have on people are great learning skills' should be 'computers have great learning skills'."}, 'sentence_structure': {'score': 4, 'feedback': 'The essay has a lot of simple sentences and lacks varied sentence structures. Consider using compound or complex sentences to improve the flow of the essay.'}, 'coherence': {'score': 7, 'feedback': 'The essay has a clear main idea, but the transitions between ideas could be smoother. Consider using transitional phrases to connect your ideas.'}, 'argument_strength': {'score': 5, 'feedback': 'The essay presents a clear argument, but it could be stronger with more evidence and analysis. Consider incorporating more examples and counterarguments to build a more persuasive argument.'}, 'vocabulary': {'score': 8, 'feedback': "The essay uses a 

In [44]:
print("Grammar Score:", feedback["grammar"]["score"])
print("Coherence Score:", feedback["coherence"]["score"])
print("Overall Score:", feedback["overall"]["score"])

print("\nSuggestions:")
for suggestion in feedback["suggestions"]:
    print("-", suggestion)

Grammar Score: 6
Coherence Score: 7
Overall Score: 5

Suggestions:
- Consider adding more evidence and analysis to support your argument.
- Revise and edit the essay to improve sentence structure and coherence.
- Use more precise and nuanced vocabulary to add depth and complexity to the essay.


In [45]:
def evaluate_essay(essay_text, essay_set):

    # 1. Clean essay
    clean_text = clean_essay(essay_text)

    # 2. Extract NLP features
    features = analyze_essay(clean_text)

    # 3. Create numeric features
    numeric_values = pd.DataFrame([[
        features['word_count'],
        features['sentence_count'],
        features['avg_sentence_length'],
        features['unique_word_count'],
        features['vocabulary_richness'],
        textstat.flesch_reading_ease(clean_text),
        features['comma_count'],
        features['question_count'],
        features['exclamation_count'],
        features['long_sentence_count'],
        features['repeated_word_count']
    ]], columns=numeric_features)

    # 4. Scale numeric features
    numeric_values = scaler.transform(numeric_values)

    # 5. TF-IDF features
    essay_tfidf = vectorizer.transform([clean_text])

    # 6. Combine TF-IDF + NLP features
    essay_combined = hstack([
        essay_tfidf,
        numeric_values
    ])

    # 7. Select the correct essay-set model
    if essay_set not in essay_set_models:
        raise ValueError(
            f"Invalid essay set. Choose one of: "
            f"{list(essay_set_models.keys())}"
        )

    selected_model = essay_set_models[essay_set]

    # 8. Predict ML score
    predicted_score = selected_model.predict(
        essay_combined
    )[0]

    # Keep prediction between 0 and 1
    predicted_score = max(0, min(1, predicted_score))

    predicted_percentage = predicted_score * 100

    # 9. Generate AI feedback
    ai_response = ollama.chat(
        model='llama3.2:3b',
        messages=[
            {
                'role': 'user',
                'content': f"""
You are an expert English writing evaluator.

Analyze the student essay and return ONLY valid JSON.

Use exactly this structure:

{{
  "grammar": {{
    "score": 0,
    "feedback": ""
  }},
  "sentence_structure": {{
    "score": 0,
    "feedback": ""
  }},
  "coherence": {{
    "score": 0,
    "feedback": ""
  }},
  "argument_strength": {{
    "score": 0,
    "feedback": ""
  }},
  "vocabulary": {{
    "score": 0,
    "feedback": ""
  }},
  "overall": {{
    "score": 0,
    "feedback": ""
  }},
  "suggestions": [
    "",
    "",
    ""
  ]
}}

Rules:
- Scores must be between 1 and 10.
- Give concise and constructive feedback.
- Suggestions must be specific and actionable.
- Do not invent grammar errors.
- Only report a grammar error if there is a clear grammatical mistake.
- Distinguish grammar errors from style suggestions.
- Do not include Markdown.
- Do not include anything outside the JSON.

Student Essay:
{clean_text}
"""
            }
        ]
    )

    # 10. Convert AI response to dictionary
    feedback_text = ai_response['message']['content']

    feedback = json.loads(feedback_text)

    # 11. Add ML prediction
    feedback["ml_predicted_score"] = round(
        predicted_percentage,
        2
    )

    return feedback

In [65]:
new_essay = """
Technology has become an essential part of modern education. It has changed the way students learn, communicate, and access information. With computers, smartphones, and online learning platforms, students can now study from almost anywhere and find educational resources within seconds.

One major advantage of technology is that it makes learning more accessible. Students can watch educational videos, attend virtual classes, read digital books, and use interactive applications to understand difficult concepts. For example, a student who struggles with mathematics can use online tutorials and practice platforms to learn a topic at their own pace. This flexibility can make education more engaging and effective.

However, technology also has some disadvantages when it is used excessively. Spending too much time on smartphones and social media can distract students from their studies and reduce their ability to concentrate. In addition, depending entirely on technology may prevent students from developing important skills such as critical thinking, communication, and independent problem-solving.

Therefore, technology should be used as a tool to support education rather than replace traditional learning completely. Teachers should encourage students to use digital resources responsibly while also giving them opportunities to participate in discussions, write independently, and work with their classmates. A balanced approach can provide students with the benefits of technology without allowing it to become a distraction.

In conclusion, technology has the potential to greatly improve education when it is used appropriately. By combining digital resources with traditional teaching methods, students can enjoy a more flexible, engaging, and effective learning experience.
"""
result = evaluate_essay(new_essay, 1)

print(json.dumps(result, indent=2))

{
  "grammar": {
    "score": 9,
    "feedback": ""
  },
  "sentence_structure": {
    "score": 8,
    "feedback": ""
  },
  "coherence": {
    "score": 9,
    "feedback": ""
  },
  "argument_strength": {
    "score": 8,
    "feedback": ""
  },
  "vocabulary": {
    "score": 8,
    "feedback": ""
  },
  "overall": {
    "score": 8,
    "feedback": ""
  },
  "suggestions": [
    "Consider adding more nuanced analysis of the benefits and drawbacks of technology in education.",
    "The essay could benefit from more specific examples to support the claim that technology should be used as a tool to support education rather than replace traditional learning.",
    "The conclusion could be strengthened by summarizing the main points more effectively."
  ],
  "ml_predicted_score": 68.43
}


In [48]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

gb_model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

gb_model.fit(X_train_numeric, y_train)

y_pred_gb = gb_model.predict(X_test_numeric)

mae_gb = mean_absolute_error(y_test, y_pred_gb)
rmse_gb = np.sqrt(mean_squared_error(y_test, y_pred_gb))
correlation_gb = np.corrcoef(y_test, y_pred_gb)[0, 1]

print("Gradient Boosting Results")
print("-------------------------")
print("MAE:", mae_gb)
print("RMSE:", rmse_gb)
print("Correlation:", correlation_gb)

Gradient Boosting Results
-------------------------
MAE: 0.1364962072685447
RMSE: 0.17696815931336735
Correlation: 0.679413684714769


In [66]:
new_essay = """
Technology has changed the way students learn.
Students can now access educational resources from anywhere.
However, excessive use of technology can reduce concentration.
Therefore, technology should be used carefully in education.
"""

result = evaluate_essay(new_essay,1)

print(json.dumps(result, indent=2))

{
  "grammar": {
    "score": 9,
    "feedback": ""
  },
  "sentence_structure": {
    "score": 9,
    "feedback": ""
  },
  "coherence": {
    "score": 9,
    "feedback": ""
  },
  "argument_strength": {
    "score": 8,
    "feedback": "More specific examples to support the claim would strengthen the argument."
  },
  "vocabulary": {
    "score": 7,
    "feedback": "Consider using more precise terms to describe the impact of excessive technology use."
  },
  "overall": {
    "score": 8,
    "feedback": "A more nuanced discussion of the benefits and drawbacks of technology use in education could enhance the essay."
  },
  "suggestions": [
    "Consider adding transitional phrases to improve the flow of ideas between sentences.",
    "Elaborate on the potential consequences of excessive technology use to make the argument more compelling.",
    "Use concrete examples to illustrate the importance of careful technology use in education."
  ],
  "ml_predicted_score": 27.72
}


In [68]:
print("Correlation with word count:")
print(df['word_count'].corr(df['domain1_score']))

print("\nCorrelation with sentence count:")
print(df['sentence_count'].corr(df['domain1_score']))

print("\nCorrelation with vocabulary richness:")
print(df['vocabulary_richness'].corr(df['domain1_score']))

Correlation with word count:
0.5767380497318282

Correlation with sentence count:
0.6101594526141364

Correlation with vocabulary richness:
-0.396016399770289


In [69]:
def advanced_features(text):
    sentences = re.split(r'[.!?]+', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    
    words = text.split()
    word_list = [
        w.lower().strip(".,!?;:\"'()")
        for w in words
    ]
    word_list = [w for w in word_list if w]

    # Sentence lengths
    sentence_lengths = [len(s.split()) for s in sentences]

    # Transition words
    transition_words = [
        'however', 'therefore', 'moreover', 'furthermore',
        'although', 'because', 'thus', 'consequently',
        'additionally', 'firstly', 'secondly', 'finally',
        'for example', 'for instance', 'in conclusion'
    ]

    transition_count = sum(
        text.lower().count(word)
        for word in transition_words
    )

    # Paragraph count
    paragraph_count = len([
        p for p in text.split('\n')
        if p.strip()
    ])

    # Average word length
    avg_word_length = (
        sum(len(w) for w in word_list) / len(word_list)
        if word_list else 0
    )

    # Sentence length variation
    sentence_length_std = (
        np.std(sentence_lengths)
        if sentence_lengths else 0
    )

    # Punctuation density
    punctuation_count = sum(
        text.count(p)
        for p in [',', '.', ';', ':', '!', '?']
    )

    punctuation_density = (
        punctuation_count / len(words)
        if words else 0
    )

    # Transition density
    transition_density = (
        transition_count / len(words)
        if words else 0
    )

    return {
        'avg_word_length': avg_word_length,
        'sentence_length_std': sentence_length_std,
        'paragraph_count': paragraph_count,
        'transition_count': transition_count,
        'transition_density': transition_density,
        'punctuation_density': punctuation_density
    }

In [70]:
advanced_df = (
    df['clean_essay']
    .apply(advanced_features)
    .apply(pd.Series)
)

df = pd.concat([df, advanced_df], axis=1)

print(df[[
    'avg_word_length',
    'sentence_length_std',
    'paragraph_count',
    'transition_count',
    'transition_density',
    'punctuation_density'
]].head())

   avg_word_length  sentence_length_std  paragraph_count  transition_count  \
0         4.356287            13.131397              1.0               4.0   
1         4.325183            11.608187              1.0               1.0   
2         4.371324            13.404424              1.0               0.0   
3         4.712810            10.477227              1.0               5.0   
4         4.405640             6.616142              1.0               4.0   

   transition_density  punctuation_density  
0            0.011869             0.103858  
1            0.002421             0.077482  
2            0.000000             0.084249  
3            0.010246             0.081967  
4            0.008658             0.093074  


In [71]:
print(
    df[[
        'avg_word_length',
        'sentence_length_std',
        'paragraph_count',
        'transition_count',
        'transition_density',
        'punctuation_density',
        'domain1_score'
    ]].corr()['domain1_score'].sort_values(ascending=False)
)

domain1_score          1.000000
transition_count       0.181170
sentence_length_std    0.097295
punctuation_density    0.055809
transition_density    -0.151836
avg_word_length       -0.372469
paragraph_count             NaN
Name: domain1_score, dtype: float64


In [72]:
from scipy.sparse import hstack
from sklearn.preprocessing import OneHotEncoder

# Encode essay set
encoder = OneHotEncoder(
    sparse_output=True,
    handle_unknown='ignore'
)

train_set = encoder.fit_transform(
    df.loc[X_train.index, ['essay_set']]
)

test_set = encoder.transform(
    df.loc[X_test.index, ['essay_set']]
)

print("Essay set features:", train_set.shape)

Essay set features: (10380, 8)


In [73]:
X_train_with_set = hstack([
    X_train_tfidf,
    X_train_numeric,
    train_set
])

X_test_with_set = hstack([
    X_test_tfidf,
    X_test_numeric,
    test_set
])

print("Training shape:", X_train_with_set.shape)
print("Testing shape:", X_test_with_set.shape)

Training shape: (10380, 10019)
Testing shape: (2596, 10019)


In [74]:
from sklearn.linear_model import Ridge

ridge_set_model = Ridge(alpha=1.0)

ridge_set_model.fit(
    X_train_with_set,
    y_train
)

y_pred_set = ridge_set_model.predict(
    X_test_with_set
)

In [75]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae_set = mean_absolute_error(y_test, y_pred_set)

rmse_set = np.sqrt(
    mean_squared_error(y_test, y_pred_set)
)

correlation_set = np.corrcoef(
    y_test,
    y_pred_set
)[0, 1]

print("Ridge + TF-IDF + NLP + Essay Set")
print("--------------------------------")
print("MAE:", mae_set)
print("RMSE:", rmse_set)
print("Correlation:", correlation_set)

Ridge + TF-IDF + NLP + Essay Set
--------------------------------
MAE: 0.11124077092090466
RMSE: 0.14611441017265792
Correlation: 0.7962311459386585


In [76]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

alphas = [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0]

results = []

for alpha in alphas:

    model = Ridge(alpha=alpha)

    model.fit(
        X_train_with_set,
        y_train
    )

    predictions = model.predict(
        X_test_with_set
    )

    mae = mean_absolute_error(y_test, predictions)

    rmse = np.sqrt(
        mean_squared_error(y_test, predictions)
    )

    correlation = np.corrcoef(
        y_test,
        predictions
    )[0, 1]

    results.append({
        "alpha": alpha,
        "MAE": mae,
        "RMSE": rmse,
        "Correlation": correlation
    })

results_df = pd.DataFrame(results)

print(results_df)

   alpha       MAE      RMSE  Correlation
0   0.01  0.195944  0.247701     0.571902
1   0.10  0.134187  0.172966     0.725760
2   0.50  0.115026  0.150876     0.783055
3   1.00  0.111241  0.146114     0.796231
4   2.00  0.109416  0.143501     0.803691
5   5.00  0.109355  0.142833     0.805935
6  10.00  0.110703  0.144205     0.802372
7  20.00  0.112926  0.147008     0.794641


In [77]:
from sklearn.linear_model import Ridge

combined_model = Ridge(alpha=5.0)

combined_model.fit(
    X_train_with_set,
    y_train
)

print("Final Ridge model trained successfully!")

Final Ridge model trained successfully!


In [78]:
print(combined_model.n_features_in_)

10019


In [79]:
new_essay = """
Technology has become an essential part of modern education. It has changed the way students learn, communicate, and access information. With computers, smartphones, and online learning platforms, students can now study from almost anywhere and find educational resources within seconds.

One major advantage of technology is that it makes learning more accessible. Students can watch educational videos, attend virtual classes, read digital books, and use interactive applications to understand difficult concepts. For example, a student who struggles with mathematics can use online tutorials and practice platforms to learn a topic at their own pace. This flexibility can make education more engaging and effective.

However, technology also has some disadvantages when it is used excessively. Spending too much time on smartphones and social media can distract students from their studies and reduce their ability to concentrate. In addition, depending entirely on technology may prevent students from developing important skills such as critical thinking, communication, and independent problem-solving.

Therefore, technology should be used as a tool to support education rather than replace traditional learning completely. Teachers should encourage students to use digital resources responsibly while also giving them opportunities to participate in discussions, write independently, and work with their classmates. A balanced approach can provide students with the benefits of technology without allowing it to become a distraction.

In conclusion, technology has the potential to greatly improve education when it is used appropriately. By combining digital resources with traditional teaching methods, students can enjoy a more flexible, engaging, and effective learning experience.
"""

result = evaluate_essay(new_essay, 1)

print(json.dumps(result, indent=2))

{
  "grammar": {
    "score": 9,
    "feedback": ""
  },
  "sentence_structure": {
    "score": 8,
    "feedback": ""
  },
  "coherence": {
    "score": 9,
    "feedback": ""
  },
  "argument_strength": {
    "score": 8,
    "feedback": ""
  },
  "vocabulary": {
    "score": 7,
    "feedback": "Some phrases, such as 'makes learning more accessible' and 'has the potential to greatly improve education,' are a bit generic. Consider using more specific examples or vivid language to support your claims."
  },
  "overall": {
    "score": 8,
    "feedback": "The essay provides a clear and concise argument about the role of technology in education. With some refinement in vocabulary and sentence structure, it has the potential to be even more effective."
  },
  "suggestions": [
    "Consider adding more specific examples or anecdotes to support your claims about the benefits and drawbacks of technology in education.",
    "Vary sentence structure by using a mix of short and long sentences, as 

In [80]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd

set_results = []

for essay_set in sorted(df['essay_set'].unique()):

    # Get train and test rows for this essay set
    train_idx = X_train.index[
        df.loc[X_train.index, 'essay_set'] == essay_set
    ]

    test_idx = X_test.index[
        df.loc[X_test.index, 'essay_set'] == essay_set
    ]

    # TF-IDF
    X_set_train = vectorizer.transform(
        df.loc[train_idx, 'clean_essay']
    )

    X_set_test = vectorizer.transform(
        df.loc[test_idx, 'clean_essay']
    )

    # Numeric features
    X_num_train = scaler.transform(
        df.loc[train_idx, numeric_features]
    )

    X_num_test = scaler.transform(
        df.loc[test_idx, numeric_features]
    )

    # Combine
    X_set_train_combined = hstack([
        X_set_train,
        X_num_train
    ])

    X_set_test_combined = hstack([
        X_set_test,
        X_num_test
    ])

    # Train model
    model = Ridge(alpha=5.0)

    model.fit(
        X_set_train_combined,
        y_train.loc[train_idx]
    )

    # Predict
    predictions = model.predict(
        X_set_test_combined
    )

    # Metrics
    mae = mean_absolute_error(
        y_test.loc[test_idx],
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test.loc[test_idx],
            predictions
        )
    )

    correlation = np.corrcoef(
        y_test.loc[test_idx],
        predictions
    )[0, 1]

    set_results.append({
        'Essay Set': essay_set,
        'MAE': mae,
        'RMSE': rmse,
        'Correlation': correlation
    })

set_results_df = pd.DataFrame(set_results)

print(set_results_df)

   Essay Set       MAE      RMSE  Correlation
0          1  0.059730  0.076015     0.857656
1          2  0.078374  0.100371     0.781669
2          3  0.151264  0.194534     0.701144
3          4  0.131128  0.173535     0.831756
4          5  0.104042  0.129718     0.834317
5          6  0.103685  0.130504     0.850639
6          7  0.103306  0.127960     0.790358
7          8  0.062310  0.079121     0.758508


In [81]:
all_predictions = []
all_actual = []

for essay_set in sorted(df['essay_set'].unique()):

    train_idx = X_train.index[
        df.loc[X_train.index, 'essay_set'] == essay_set
    ]

    test_idx = X_test.index[
        df.loc[X_test.index, 'essay_set'] == essay_set
    ]

    X_set_train = vectorizer.transform(
        df.loc[train_idx, 'clean_essay']
    )

    X_set_test = vectorizer.transform(
        df.loc[test_idx, 'clean_essay']
    )

    X_num_train = scaler.transform(
        df.loc[train_idx, numeric_features]
    )

    X_num_test = scaler.transform(
        df.loc[test_idx, numeric_features]
    )

    X_set_train_combined = hstack([
        X_set_train,
        X_num_train
    ])

    X_set_test_combined = hstack([
        X_set_test,
        X_num_test
    ])

    model = Ridge(alpha=5.0)

    model.fit(
        X_set_train_combined,
        y_train.loc[train_idx]
    )

    predictions = model.predict(
        X_set_test_combined
    )

    all_predictions.extend(predictions)
    all_actual.extend(y_test.loc[test_idx])

overall_mae = mean_absolute_error(
    all_actual,
    all_predictions
)

overall_rmse = np.sqrt(
    mean_squared_error(
        all_actual,
        all_predictions
    )
)

overall_correlation = np.corrcoef(
    all_actual,
    all_predictions
)[0, 1]

print("Separate Essay-Set Models")
print("-------------------------")
print("MAE:", overall_mae)
print("RMSE:", overall_rmse)
print("Correlation:", overall_correlation)

Separate Essay-Set Models
-------------------------
MAE: 0.10158852072826247
RMSE: 0.13513856071721006
Correlation: 0.8283581945053503


In [82]:
from sklearn.linear_model import Ridge

essay_set_models = {}

for essay_set in sorted(df['essay_set'].unique()):

    # Get train and test indices
    train_idx = X_train.index[
        df.loc[X_train.index, 'essay_set'] == essay_set
    ]

    # TF-IDF features
    X_set_train = vectorizer.transform(
        df.loc[train_idx, 'clean_essay']
    )

    # Numeric features
    X_num_train = scaler.transform(
        df.loc[train_idx, numeric_features]
    )

    # Combine features
    X_set_train_combined = hstack([
        X_set_train,
        X_num_train
    ])

    # Train model
    model = Ridge(alpha=5.0)

    model.fit(
        X_set_train_combined,
        y_train.loc[train_idx]
    )

    # Store model
    essay_set_models[essay_set] = model

print("Models trained successfully!")
print("Number of models:", len(essay_set_models))
print("Available essay sets:", list(essay_set_models.keys()))

Models trained successfully!
Number of models: 8
Available essay sets: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]


In [83]:
import joblib

joblib.dump(vectorizer, "tfidf_vectorizer.pkl")
joblib.dump(scaler, "feature_scaler.pkl")
joblib.dump(essay_set_models, "essay_set_models.pkl")

print("All ML components saved successfully!")

All ML components saved successfully!


In [84]:
import joblib

all_models = {
    "Ridge - TF-IDF": model,
    "Ridge - TF-IDF + NLP Features": combined_model,
    "Random Forest": rf_model,
    "Gradient Boosting": gb_model,
    "Ridge - Separate Essay Set": essay_set_models
}

joblib.dump(all_models, "all_models.pkl")

print("All models saved successfully!")

All models saved successfully!


In [86]:
import joblib

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from scipy.sparse import hstack


# ============================================================
# 1. RIDGE - TF-IDF ONLY
# Expected input features = 10,000
# ============================================================

ridge_tfidf_model = Ridge(alpha=1.0)

ridge_tfidf_model.fit(
    X_train_tfidf,
    y_train
)

print("Ridge TF-IDF trained")


# ============================================================
# 2. RIDGE - TF-IDF + NLP FEATURES
# Expected input features = 10,011
# ============================================================

ridge_nlp_model = Ridge(alpha=1.0)

ridge_nlp_model.fit(
    X_train_combined,
    y_train
)

print("Ridge TF-IDF + NLP trained")


# ============================================================
# 3. RANDOM FOREST
# Expected input features = 11
# ============================================================

rf_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(
    X_train_numeric,
    y_train
)

print("Random Forest trained")


# ============================================================
# 4. GRADIENT BOOSTING
# Expected input features = 11
# ============================================================

gb_model = GradientBoostingRegressor(
    n_estimators=100,
    random_state=42
)

gb_model.fit(
    X_train_numeric,
    y_train
)

print("Gradient Boosting trained")


# ============================================================
# 5. SEPARATE RIDGE MODEL FOR EACH ESSAY SET
# Expected input features = 10,011
# ============================================================

essay_set_models = {}

for essay_set in sorted(df['essay_set'].unique()):

    # Get training rows belonging to this essay set
    train_idx = X_train.index[
        df.loc[X_train.index, 'essay_set'] == essay_set
    ]

    # TF-IDF
    X_set_train_tfidf = vectorizer.transform(
        df.loc[train_idx, 'clean_essay']
    )

    # Numeric features
    X_set_train_numeric = scaler.transform(
        df.loc[train_idx, numeric_features]
    )

    # Combine TF-IDF + NLP
    X_set_train_combined = hstack([
        X_set_train_tfidf,
        X_set_train_numeric
    ])

    # Separate Ridge model
    set_model = Ridge(alpha=5.0)

    set_model.fit(
        X_set_train_combined,
        y_train.loc[train_idx]
    )

    essay_set_models[essay_set] = set_model

print("Separate Essay Set models trained")
print("Available essay sets:", list(essay_set_models.keys()))


# ============================================================
# 6. SAVE INDIVIDUAL COMPONENTS
# ============================================================

joblib.dump(
    vectorizer,
    "tfidf_vectorizer.pkl"
)

joblib.dump(
    scaler,
    "feature_scaler.pkl"
)

joblib.dump(
    essay_set_models,
    "essay_set_models.pkl"
)


# ============================================================
# 7. SAVE ALL MODELS CORRECTLY
# ============================================================

all_models = {

    "Ridge - TF-IDF":
        ridge_tfidf_model,

    "Ridge - TF-IDF + NLP Features":
        ridge_nlp_model,

    "Random Forest":
        rf_model,

    "Gradient Boosting":
        gb_model,

    "Ridge - Separate Essay Set":
        essay_set_models
}


joblib.dump(
    all_models,
    "all_models.pkl"
)


print("\n====================================")
print("ALL MODELS SAVED SUCCESSFULLY!")
print("====================================")

print("\nModel feature counts:")

print(
    "Ridge - TF-IDF:",
    ridge_tfidf_model.n_features_in_
)

print(
    "Ridge - TF-IDF + NLP Features:",
    ridge_nlp_model.n_features_in_
)

print(
    "Random Forest:",
    rf_model.n_features_in_
)

print(
    "Gradient Boosting:",
    gb_model.n_features_in_
)

print(
    "Separate Ridge models: 10,011 features each"
)

print("\nFiles created:")
print("tfidf_vectorizer.pkl")
print("feature_scaler.pkl")
print("essay_set_models.pkl")
print("all_models.pkl")


Ridge TF-IDF trained
Ridge TF-IDF + NLP trained
Random Forest trained
Gradient Boosting trained
Separate Essay Set models trained
Available essay sets: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]

ALL MODELS SAVED SUCCESSFULLY!

Model feature counts:
Ridge - TF-IDF: 10000
Ridge - TF-IDF + NLP Features: 10011
Random Forest: 11
Gradient Boosting: 11
Separate Ridge models: 10,011 features each

Files created:
tfidf_vectorizer.pkl
feature_scaler.pkl
essay_set_models.pkl
all_models.pkl
